# 6.3 Metadata Trial 004: Threshold Optimization

Purpose:
- Evaluate different probability thresholds.
- Choose a threshold suitable for contextual stroke-risk screening.
- Avoid blindly using the default 0.5 threshold.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from rural_stroke_assist.config import (
    METADATA_SPLIT_MANIFEST_PATH,
    METADATA_TRIAL_001_MODEL_PATH,
    METADATA_TRIAL_004_THRESHOLD_SUMMARY_PATH,
    METADATA_TRIAL_004_RESULTS_PATH,
    METADATA_MVP_MODEL_PATH,
)

from rural_stroke_assist.preprocessing.metadata import (
    prepare_metadata_modeling_frame,
    split_metadata_features_target,
)

from rural_stroke_assist.modeling.metadata_baselines import (
    load_metadata_model,
    save_metadata_model,
)

from rural_stroke_assist.modeling.evaluation import (
    evaluate_binary_classifier,
    create_classification_report_dict,
)

In [2]:
metadata_df = pd.read_csv(METADATA_SPLIT_MANIFEST_PATH)
metadata_model_df = prepare_metadata_modeling_frame(metadata_df)

val_df = metadata_model_df[metadata_model_df["split"] == "val"].copy()
test_df = metadata_model_df[metadata_model_df["split"] == "test"].copy()

X_val, y_val = split_metadata_features_target(val_df)
X_test, y_test = split_metadata_features_target(test_df)

In [3]:
model = load_metadata_model(METADATA_TRIAL_001_MODEL_PATH)

val_scores = model.predict_proba(X_val)[:, 1]
test_scores = model.predict_proba(X_test)[:, 1]

val_scores[:10]

array([0.50397222, 0.90293722, 0.64014007, 0.74575065, 0.42537544,
       0.79947838, 0.8617452 , 0.47907437, 0.92364127, 0.43592041])

In [4]:
thresholds = np.arange(0.05, 0.95, 0.05)

rows = []

for threshold in thresholds:
    val_pred = (val_scores >= threshold).astype(int)

    metrics = evaluate_binary_classifier(
        y_true=y_val.to_numpy(),
        y_pred=val_pred,
        y_score=val_scores,
    )

    report = create_classification_report_dict(
        y_true=y_val.to_numpy(),
        y_pred=val_pred,
        target_names=["no_stroke", "stroke"],
    )

    rows.append(
        {
            "threshold": threshold,
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
            "roc_auc": metrics["roc_auc"],
            "stroke_precision": report["stroke"]["precision"],
            "stroke_recall": report["stroke"]["recall"],
            "stroke_f1": report["stroke"]["f1-score"],
            "predicted_positive_count": int(val_pred.sum()),
        }
    )

threshold_df = pd.DataFrame(rows)

threshold_df

,threshold,accuracy,precision,recall,f1,roc_auc,stroke_precision,stroke_recall,stroke_f1,predicted_positive_count
0,0.05,0.283290,0.063140,1.000000,0.118780,0.845179,0.063140,1.000000,0.118780,586
1,0.10,0.402089,0.073022,0.972973,0.135849,0.845179,0.073022,0.972973,0.135849,493
2,0.15,0.468668,0.079545,0.945946,0.146751,0.845179,0.079545,0.945946,0.146751,440
3,0.20,0.530026,0.086957,0.918919,0.158879,0.845179,0.086957,0.918919,0.158879,391
4,0.25,0.573107,0.094972,0.918919,0.172152,0.845179,0.094972,0.918919,0.172152,358
5,0.30,0.613577,0.101538,0.891892,0.182320,0.845179,0.101538,0.891892,0.182320,325
6,0.35,0.648825,0.110738,0.891892,0.197015,0.845179,0.110738,0.891892,0.197015,298
7,0.40,0.684073,0.116105,0.837838,0.203947,0.845179,0.116105,0.837838,0.203947,267
8,0.45,0.707572,0.118367,0.783784,0.205674,0.845179,0.118367,0.783784,0.205674,245
9,0.50,0.737598,0.127273,0.756757,0.217899,0.845179,0.127273,0.756757,0.217899,220


In [5]:
candidate_thresholds = threshold_df[
    threshold_df["stroke_recall"] >= 0.75
].copy()

if len(candidate_thresholds) > 0:
    best_threshold_row = candidate_thresholds.sort_values(
        by=["stroke_f1", "stroke_precision"],
        ascending=False,
    ).iloc[0]
else:
    best_threshold_row = threshold_df.sort_values(
        by=["stroke_f1"],
        ascending=False,
    ).iloc[0]

best_threshold = float(best_threshold_row["threshold"])

best_threshold_row

threshold                     0.500000
accuracy                      0.737598
precision                     0.127273
recall                        0.756757
f1                            0.217899
roc_auc                       0.845179
stroke_precision              0.127273
stroke_recall                 0.756757
stroke_f1                     0.217899
predicted_positive_count    220.000000
Name: 9, dtype: float64

In [6]:
test_pred = (test_scores >= best_threshold).astype(int)

test_metrics = evaluate_binary_classifier(
    y_true=y_test.to_numpy(),
    y_pred=test_pred,
    y_score=test_scores,
)

test_report = create_classification_report_dict(
    y_true=y_test.to_numpy(),
    y_pred=test_pred,
    target_names=["no_stroke", "stroke"],
)

test_metrics, test_report

({'accuracy': 0.7235984354628422,
  'precision': 0.13135593220338984,
  'recall': 0.8157894736842105,
  'f1': 0.22627737226277372,
  'roc_auc': 0.8337665150530646},
 {'no_stroke': {'precision': 0.9868173258003766,
   'recall': 0.7187928669410151,
   'f1-score': 0.8317460317460318,
   'support': 729.0},
  'stroke': {'precision': 0.13135593220338984,
   'recall': 0.8157894736842105,
   'f1-score': 0.22627737226277372,
   'support': 38.0},
  'accuracy': 0.7235984354628422,
  'macro avg': {'precision': 0.5590866290018832,
   'recall': 0.7672911703126128,
   'f1-score': 0.5290117020044027,
   'support': 767.0},
  'weighted avg': {'precision': 0.9444346231189092,
   'recall': 0.7235984354628422,
   'f1-score': 0.8017488882514245,
   'support': 767.0}})

In [7]:
METADATA_TRIAL_004_THRESHOLD_SUMMARY_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

threshold_df.to_csv(
    METADATA_TRIAL_004_THRESHOLD_SUMMARY_PATH,
    index=False,
)

METADATA_TRIAL_004_THRESHOLD_SUMMARY_PATH

WindowsPath('C:/Users/Asus/Downloads/Uni_work/Grad-Project/rural-stroke-assist/data/processed/experiments/metadata_trial_004_threshold_summary.csv')

In [8]:
stroke_recall = test_report["stroke"]["recall"]
stroke_precision = test_report["stroke"]["precision"]
stroke_f1 = test_report["stroke"]["f1-score"]
roc_auc = test_metrics["roc_auc"]

accepted = (
    stroke_recall >= 0.75
    and roc_auc >= 0.80
)

{
    "best_threshold": best_threshold,
    "stroke_precision": stroke_precision,
    "stroke_recall": stroke_recall,
    "stroke_f1": stroke_f1,
    "roc_auc": roc_auc,
    "accepted": accepted,
}

{'best_threshold': 0.5,
 'stroke_precision': 0.13135593220338984,
 'stroke_recall': 0.8157894736842105,
 'stroke_f1': 0.22627737226277372,
 'roc_auc': 0.8337665150530646,
 'accepted': True}

In [9]:
report = f"""# Metadata Trial 004 Threshold Optimization

## Purpose

This experiment evaluates probability thresholds for the accepted metadata risk-factor model.

## Selected Threshold

Selected threshold: {best_threshold:.2f}

Selection logic:
- Prefer thresholds with stroke recall >= 0.75.
- Among those, choose the threshold with highest stroke F1.

## Test Performance at Selected Threshold

- Accuracy: {test_metrics["accuracy"]:.4f}
- Stroke precision: {stroke_precision:.4f}
- Stroke recall: {stroke_recall:.4f}
- Stroke F1: {stroke_f1:.4f}
- ROC-AUC: {roc_auc:.4f}

## Interpretation

The metadata model is accepted as a contextual risk-factor model, not as an acute triage model.

Its low precision reflects the severe class imbalance and the fact that available metadata contains risk factors rather than acute neurological symptoms.

## Decision

Accepted for MVP as a weak contextual risk component.

It should have a lower fusion weight than face and speech, and should later be supplemented by a FAST/NIHSS-inspired structured questionnaire.
"""

METADATA_TRIAL_004_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
METADATA_TRIAL_004_RESULTS_PATH.write_text(report, encoding="utf-8")

print(report)

# Metadata Trial 004 Threshold Optimization

## Purpose

This experiment evaluates probability thresholds for the accepted metadata risk-factor model.

## Selected Threshold

Selected threshold: 0.50

Selection logic:
- Prefer thresholds with stroke recall >= 0.75.
- Among those, choose the threshold with highest stroke F1.

## Test Performance at Selected Threshold

- Accuracy: 0.7236
- Stroke precision: 0.1314
- Stroke recall: 0.8158
- Stroke F1: 0.2263
- ROC-AUC: 0.8338

## Interpretation

The metadata model is accepted as a contextual risk-factor model, not as an acute triage model.

Its low precision reflects the severe class imbalance and the fact that available metadata contains risk factors rather than acute neurological symptoms.

## Decision

Accepted for MVP as a weak contextual risk component.

It should have a lower fusion weight than face and speech, and should later be supplemented by a FAST/NIHSS-inspired structured questionnaire.



In [10]:
if accepted:
    save_metadata_model(model, METADATA_MVP_MODEL_PATH)
    print("Metadata MVP model saved to:", METADATA_MVP_MODEL_PATH)
else:
    print("Metadata model not accepted. Results saved.")

Metadata MVP model saved to: C:\Users\Asus\Downloads\Uni_work\Grad-Project\rural-stroke-assist\models\experiments\metadata\mvp_metadata_risk_model.pkl
